In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

DATA_DIR = Path("/var/threat-intel/datasets")
CSV_FILES = sorted(DATA_DIR.rglob("*.csv"))
print(f"Found {len(CSV_FILES)} files")

Found 8 files


In [2]:
# Load everything into one DataFrame. The full dataset is ~3 GB unzipped CSV
# but only ~600 MB in memory because pandas is more efficient than raw CSV.
dfs = []
for f in CSV_FILES:
    print(f"Loading {f.name}...", end=" ")
    df = pd.read_csv(f, low_memory=False)
    print(f"shape={df.shape}")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
del dfs  # free memory immediately
print(f"\nCombined shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv... shape=(225745, 79)
shape=(286467, 79)kingHours-Afternoon-PortScan.pcap_ISCX.csv... 
shape=(191033, 79)kingHours-Morning.pcap_ISCX.csv... 
shape=(529918, 79)kingHours.pcap_ISCX.csv... 
shape=(288602, 79)orkingHours-Afternoon-Infilteration.pcap_ISCX.csv... 
shape=(170366, 79)orkingHours-Morning-WebAttacks.pcap_ISCX.csv... 
shape=(445909, 79)rkingHours.pcap_ISCX.csv... 
shape=(692703, 79)workingHours.pcap_ISCX.csv... 

Combined shape: (2830743, 79)
Memory: 1855.5 MB


In [3]:
# The dataset has leading spaces in column names (' Destination Port', etc).
# Strip them once at load time so we don't have to remember forever.
df.columns = df.columns.str.strip()
print("Column names (first 10):")
print(df.columns[:10].tolist())
print(f"\nLabel column: '{df.columns[-1]}'")

Column names (first 10):
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']

Label column: 'Label'


In [4]:
label_counts = df['Label'].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nTotal flows: {len(df):,}")
print(f"Benign fraction: {label_counts['BENIGN'] / len(df):.1%}")
print(f"Attack classes: {(label_counts.index != 'BENIGN').sum()}")

Label distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Total flows: 2,830,743
Benign fraction: 80.3%
Attack classes: 14


In [5]:
# Find rows with missing or infinite values — common in flow data
# where duration can be zero, causing division-by-zero in derived features
nan_counts = df.isna().sum()
inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()

print("Columns with NaN values:")
print(nan_counts[nan_counts > 0])
print(f"\nColumns with Inf values:")
print(inf_counts[inf_counts > 0])

# Total rows affected
rows_with_nan = df.isna().any(axis=1).sum()
rows_with_inf = np.isinf(df.select_dtypes(include=[np.number])).any(axis=1).sum()
print(f"\nRows with at least one NaN: {rows_with_nan:,}")
print(f"Rows with at least one Inf: {rows_with_inf:,}")

Columns with NaN values:
Flow Bytes/s    1358
dtype: int64

Columns with Inf values:
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64

Rows with at least one NaN: 1,358
Rows with at least one Inf: 2,867


In [6]:
# CRITICAL: prove the destination port leak empirically.
# If destination port is a perfect predictor of attack class, the model
# learns "port 80 = web attack" instead of learning attack behavior.

print("=" * 60)
print("Destination Port distribution by attack type (top 5 ports each)")
print("=" * 60)
for label in df['Label'].unique():
    subset = df[df['Label'] == label]
    if len(subset) < 100:
        continue  # skip tiny classes for readability
    top_ports = subset['Destination Port'].value_counts().head(5)
    concentration = top_ports.iloc[0] / len(subset) * 100
    print(f"\n{label}: ({len(subset):,} flows)")
    print(f"  Top port concentration: {concentration:.1f}% on port {top_ports.index[0]}")
    print(f"  {top_ports.to_dict()}")

Destination Port distribution by attack type (top 5 ports each)

BENIGN: (2,273,097 flows)
  Top port concentration: 42.1% on port 53
  {53: 957812, 443: 505470, 80: 235695, 123: 23879, 22: 10801}

DDoS: (128,027 flows)
  Top port concentration: 100.0% on port 80
  {80: 128024, 64869: 1, 64873: 1, 27636: 1}

PortScan: (158,930 flows)
  Top port concentration: 0.2% on port 80
  {80: 373, 21: 244, 22: 243, 443: 240, 444: 209}

Bot: (1,966 flows)
  Top port concentration: 64.1% on port 8080
  {8080: 1261, 51800: 1, 53730: 1, 4089: 1, 52901: 1}

Web Attack � Brute Force: (1,507 flows)
  Top port concentration: 100.0% on port 80
  {80: 1507}

Web Attack � XSS: (652 flows)
  Top port concentration: 100.0% on port 80
  {80: 652}

FTP-Patator: (7,938 flows)
  Top port concentration: 100.0% on port 21
  {21: 7937, 80: 1}

SSH-Patator: (5,897 flows)
  Top port concentration: 100.0% on port 22
  {22: 5897}

DoS slowloris: (5,796 flows)
  Top port concentration: 100.0% on port 80
  {80: 5796}

DoS

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Simplify: classify any attack vs benign using ONLY destination port
df_test = df.copy()
df_test['is_attack'] = (df_test['Label'] != 'BENIGN').astype(int)

X_one = df_test[['Destination Port']]
y_one = df_test['is_attack']

X_train, X_test, y_train, y_test = train_test_split(
    X_one, y_one, test_size=0.2, stratify=y_one, random_state=42
)

clf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

print(f"Classifier using ONLY Destination Port:")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"\nClassification report:")
print(classification_report(y_test, preds, target_names=['BENIGN', 'ATTACK']))

Classifier using ONLY Destination Port:
Accuracy: 0.8876

Classification report:
              precision    recall  f1-score   support

      BENIGN       1.00      0.86      0.93    454620
      ATTACK       0.64      0.98      0.77    111529

    accuracy                           0.89    566149
   macro avg       0.82      0.92      0.85    566149
weighted avg       0.93      0.89      0.90    566149



In [8]:
print("=" * 60)
print("LEAKAGE DIAGNOSIS SUMMARY")
print("=" * 60)
print(f"""
Destination Port shows STRONG label leakage for several attack types
in CICIDS2017, though the effect is heterogeneous across classes.

Evidence (from data):
- 100% of DoS Hulk samples target port 80
- 100% of SSH-Patator samples target port 22
- 100% of FTP-Patator samples target port 21
- 100% of DDoS, slowloris, slowhttptest, GoldenEye target port 80
- 100% of Web Attack samples target port 80
- HOWEVER: PortScan is spread across many ports (0.2% concentration)
- BENIGN traffic is realistically distributed across port 53, 443, 80, etc.

Quantified impact:
- A classifier using ONLY Destination Port achieves 88.76% accuracy
  on the binary attack-vs-benign task
- ATTACK recall is 0.98 — the model catches almost everything
- But ATTACK precision is only 0.64 — 36% false-positive rate
- This high recall / low precision pattern indicates the model
  predicts "attack" aggressively on common attack ports, including
  for benign flows that happen to use the same port

Mechanism:
For most DoS and brute-force families, the dataset researchers fixed
the target port. A model would learn "port 22 = SSH brute force"
rather than learning brute-force *behavior* (rapid auth attempts,
short flow duration, high SYN count). PortScan is the exception —
its varied targeting means the model has to learn real behavior.

Mitigation strategy for the production model:
- Drop 'Destination Port' from features
- Also drop 'Source Port' (likely encodes researcher tool fingerprints)
- Drop identifiers: 'Flow ID', 'Source IP', 'Destination IP', 'Timestamp'
- Train on behavioral features only:
  * Packet counts (fwd/bwd)
  * Inter-arrival time statistics
  * Packet length statistics
  * TCP flag counts (SYN, ACK, FIN, RST, PSH, URG)
  * Flow duration

Expected outcome:
With leaky features removed, the model must rely on behavioral signal.
Trained correctly, XGBoost should still reach 95%+ macro precision —
not because of the leak, but because attack behavior IS genuinely
distinguishable. Our 88.76% leak-alone baseline gives us a defensible
reference: any final model significantly above this number is learning
real behavioral patterns.
""")

LEAKAGE DIAGNOSIS SUMMARY

Destination Port shows STRONG label leakage for several attack types
in CICIDS2017, though the effect is heterogeneous across classes.

Evidence (from data):
- 100% of DoS Hulk samples target port 80
- 100% of SSH-Patator samples target port 22
- 100% of FTP-Patator samples target port 21
- 100% of DDoS, slowloris, slowhttptest, GoldenEye target port 80
- 100% of Web Attack samples target port 80
- HOWEVER: PortScan is spread across many ports (0.2% concentration)
- BENIGN traffic is realistically distributed across port 53, 443, 80, etc.

Quantified impact:
- A classifier using ONLY Destination Port achieves 88.76% accuracy
  on the binary attack-vs-benign task
- ATTACK recall is 0.98 — the model catches almost everything
- But ATTACK precision is only 0.64 — 36% false-positive rate
- This high recall / low precision pattern indicates the model
  predicts "attack" aggressively on common attack ports, including
  for benign flows that happen to use the same p

In [10]:
# Any feature where a single value perfectly predicts a class is suspicious.
# Let's find them programmatically.

from sklearn.feature_selection import mutual_info_classif

# Drop non-numeric and the label
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features = df[numeric_cols].copy()

# Clean Inf/NaN for mutual info calculation
features = features.replace([np.inf, -np.inf], np.nan)
features = features.fillna(0)

# Use a small sample for speed
sample = df.sample(n=100000, random_state=42)
y_sample = (sample['Label'] != 'BENIGN').astype(int)
X_sample = sample[numeric_cols].replace([np.inf, -np.inf], 0).fillna(0)

# Mutual information per feature
mi = mutual_info_classif(X_sample, y_sample, random_state=42)
mi_df = pd.DataFrame({
    'feature': numeric_cols,
    'mutual_info': mi
}).sort_values('mutual_info', ascending=False)

print("Top 15 features by mutual information with attack label:")
print(mi_df.head(15).to_string(index=False))
print("\nBottom 10 features (likely useless or constant):")
print(mi_df.tail(10).to_string(index=False))

Top 15 features by mutual information with attack label:
                    feature  mutual_info
     Packet Length Variance     0.334304
          Packet Length Std     0.334291
        Average Packet Size     0.334009
         Packet Length Mean     0.303369
Total Length of Bwd Packets     0.293380
          Subflow Bwd Bytes     0.293271
           Destination Port     0.284248
Total Length of Fwd Packets     0.282058
          Subflow Fwd Bytes     0.281307
     Bwd Packet Length Mean     0.281002
       Avg Bwd Segment Size     0.278664
     Init_Win_bytes_forward     0.268639
          Max Packet Length     0.260936
      Bwd Packet Length Max     0.258463
    Init_Win_bytes_backward     0.250215

Bottom 10 features (likely useless or constant):
             feature  mutual_info
       Bwd PSH Flags     0.001636
   Bwd Avg Bulk Rate     0.001125
      ECE Flag Count     0.000880
       Fwd URG Flags     0.000000
       Bwd URG Flags     0.000000
Bwd Avg Packets/Bulk     0.000000

In [11]:
# Features with no variance can't help the model — drop them
constant_features = []
near_constant_features = []
for col in numeric_cols:
    n_unique = features[col].nunique()
    if n_unique == 1:
        constant_features.append(col)
    elif n_unique == 2 and features[col].value_counts(normalize=True).iloc[0] > 0.999:
        near_constant_features.append(col)

print(f"Constant features ({len(constant_features)}):")
for c in constant_features:
    print(f"  {c}")
print(f"\nNear-constant features ({len(near_constant_features)}):")
for c in near_constant_features:
    print(f"  {c}")

Constant features (8):
  Bwd PSH Flags
  Bwd URG Flags
  Fwd Avg Bytes/Bulk
  Fwd Avg Packets/Bulk
  Fwd Avg Bulk Rate
  Bwd Avg Bytes/Bulk
  Bwd Avg Packets/Bulk
  Bwd Avg Bulk Rate

Near-constant features (4):
  Fwd URG Flags
  RST Flag Count
  CWE Flag Count
  ECE Flag Count


In [12]:
# Map fine-grained CICIDS2017 labels to production categories
# that align with our attack scripts (Part 6F)
LABEL_MAP = {
    'BENIGN': 'BENIGN',
    'PortScan': 'PortScan',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'DDoS': 'DoS',
    'SSH-Patator': 'Bruteforce',
    'FTP-Patator': 'Bruteforce',
    'Bot': 'Botnet',
    'Web Attack � Brute Force': 'WebAttack',
    'Web Attack � XSS': 'WebAttack',
    'Web Attack � Sql Injection': 'WebAttack',
    'Infiltration': 'Infiltration',
    'Heartbleed': 'Infiltration',
}

df['production_label'] = df['Label'].map(LABEL_MAP)
print("Production label distribution:")
print(df['production_label'].value_counts())
print(f"\nUnmapped labels (should be 0): {df['production_label'].isna().sum()}")

Production label distribution:
production_label
BENIGN          2273097
DoS              380688
PortScan         158930
Bruteforce        13835
WebAttack          2180
Botnet             1966
Infiltration         47
Name: count, dtype: int64

Unmapped labels (should be 0): 0


In [13]:
import json
from pathlib import Path

MODELS_DIR = Path("/home/ubuntu/threat-intel-platform/ml/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Features to DROP — leaky or useless
DROP_FEATURES = [
    # Leaky: encode target-specific identifiers, not behavior
    'Destination Port',
    'Source Port',
    'Flow ID',
    'Source IP',
    'Destination IP',
    'Timestamp',
    # Add constant features we found in Cell 10
    *constant_features,
    *near_constant_features,
]

# Save for the training notebook to consume
config = {
    'drop_features': DROP_FEATURES,
    'label_map': LABEL_MAP,
    'production_labels': sorted(df['production_label'].dropna().unique().tolist()),
    'rationale': {
        'destination_port': 'Encodes attack target (port 22 = SSH-Patator, port 80 = web attack). '
                            'Model would memorize researcher choices, not learn attack behavior.',
        'source_port': 'Also dataset-specific. Real attacks use varied source ports.',
        'identifiers': 'Flow ID, IPs, Timestamps are identifiers, not behavior.',
        'constant_features': 'Zero variance contributes no information; drops noise without info loss.',
    }
}

with open(MODELS_DIR / 'feature_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Saved feature config to {MODELS_DIR / 'feature_config.json'}")
print(f"\nFeatures being dropped: {len(DROP_FEATURES)}")
print(f"Production label classes: {len(config['production_labels'])}")

Saved feature config to /home/ubuntu/threat-intel-platform/ml/models/feature_config.json

Features being dropped: 18
Production label classes: 7
